# NB08 — Final analyses for the review
**Settings:** Accelerator **GPU T4 x2** · Internet **On**.
**Inputs:** *Your Work → `nb01-selfcontained`* (data + 24 runs), *Your Work → `nb06-mask-loss`* (Correia runs) **and**
*Your Work → `nb08-final`* (the three Ours+Correia runs trained by the first run of this notebook, whose evaluation stopped at a
too strict numerical check; if they are not attached, section 1 trains them again).
**Run:** *Save Version → Save & Run All* (about 1 h with the trained runs attached, about 2 h otherwise).

1. **Projection thresholds.** The thresholds act only in the push step, after training. Ours-stdPush is trained exactly like Ours
   but projected without constraint, so re-projecting its prototypes with other thresholds (lesion fraction >= 0.3 / 0.5 / 0.7 / 0.9,
   artifact fraction <= 0 / 0.05 / 0.2 / no limit), refitting the last layer and re-evaluating gives a threshold curve without retraining.
2. **Ours+Correia:** our full objective plus the original mask loss of Correia et al. (pixel level, batch sum, lambda = 1e-3).
3. **Undetected artifacts** in admissible and prototype cells (manual labels of Bissoto et al.), **border cells**, and the
   **provenance guarantee under +-5 px lesion-mask errors**.

In [ ]:
%%writefile artproto.py
"""
artproto.py -- shared library for the paper
"Artifact-aware prototype learning for skin-lesion classification".

Everything (data caching, lesion/artifact masks, models, losses, training,
push, evaluation, counterfactuals) lives here so that every notebook uses
exactly the same code.
"""
import os, re, json, glob, math, time, random, copy, traceback
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

IMG = 256          # cached resolution (square, as in the original project)
OUT = 224          # network input resolution
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# bit layout of the uint8 mask array
BIT_L, BIT_DC, BIT_TS, BIT_SP, BIT_INK = 1, 2, 4, 8, 16
BIT_ART = BIT_DC | BIT_TS | BIT_SP | BIT_INK
ART_BITS = {"DC": BIT_DC, "TS": BIT_TS, "SP": BIT_SP, "INK": BIT_INK}
ART_LONG = {"DC": "dark corner / vignette", "TS": "thin dark structures (hair, ruler ticks)",
            "SP": "specular highlights / gel bubbles", "INK": "ink markings / coloured patches"}
DETECTOR_VERSION = 2
# image-level presence thresholds (fraction of image area)
ART_MIN_FRAC = {"DC": 0.01, "TS": 0.003, "SP": 0.0005, "INK": 0.001}


# ----------------------------------------------------------------------------- utils
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def walk_files(root="/kaggle/input"):
    """Recursive file list. followlinks=True is REQUIRED: recent Kaggle images mount
    datasets as symlinks under /kaggle/input, which os.walk skips by default."""
    out, seen = [], set()
    for r, ds, fs in os.walk(root, followlinks=True):
        real = os.path.realpath(r)
        if real in seen:                      # guard against symlink cycles
            ds[:] = []; continue
        seen.add(real)
        for f in fs:
            out.append(os.path.join(r, f).replace(os.sep, '/'))
    return out


def load_rgb_resized(path, size=IMG):
    from PIL import Image
    im = Image.open(path)
    try:
        im.draft("RGB", (size * 2, size * 2))      # fast JPEG decoding at reduced scale
    except Exception:
        pass
    a = np.asarray(im.convert("RGB"))
    return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)


def load_mask_resized(path, size=IMG):
    from PIL import Image
    m = np.asarray(Image.open(path).convert("L"))
    m = cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
    return (m >= 128).astype(np.uint8)


def phash64(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    g = cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA)
    d = cv2.dct(g)[:8, :8].flatten()
    return d > np.median(d[1:])


def thumbs(img):
    """z-normalised 32x32 grey thumbnails of the full image and of the central 50% crop
    (used to VERIFY pHash duplicate candidates; the centre crop ignores vignettes)."""
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    out = []
    for a in (g, g[64:192, 64:192]):
        t = cv2.resize(a, (32, 32), interpolation=cv2.INTER_AREA).flatten()
        t = (t - t.mean()) / (t.std() + 1e-6) / np.sqrt(t.size)
        out.append(t.astype(np.float16))
    return out[0], out[1]


def hamming_pairs(bitsA, bitsB=None, thr=6, device="cuda", chunk=2048):
    """Return (i, j, ham) with ham <= thr. If bitsB is None: pairs inside A with i<j."""
    dev = device if torch.cuda.is_available() else "cpu"
    dt = torch.float16 if dev != "cpu" else torch.float32
    A = torch.from_numpy(bitsA.astype(np.float32) * 2 - 1).to(dev, dt)
    B = A if bitsB is None else torch.from_numpy(bitsB.astype(np.float32) * 2 - 1).to(dev, dt)
    out = []
    for s in range(0, len(A), chunk):
        ham = (64 - (A[s:s + chunk] @ B.T).float()) / 2
        ii, jj = torch.nonzero(ham <= thr + 1e-3, as_tuple=True)
        h = ham[ii, jj]
        ii = ii + s
        if bitsB is None:
            keep = ii < jj
            ii, jj, h = ii[keep], jj[keep], h[keep]
        out.append(torch.stack([ii.float(), jj.float(), h], 1).cpu())
    o = torch.cat(out).numpy() if out else np.zeros((0, 3))
    return o[:, 0].astype(int), o[:, 1].astype(int), o[:, 2].astype(int)


class UnionFind:
    def __init__(self, n): self.p = np.arange(n)
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)


# ----------------------------------------------------------------------------- masks
def postprocess_lesion(prob, thr=0.5, min_frac=0.005):
    m = (prob >= thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.zeros_like(m), False
    j = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    m = (lab == j).astype(np.uint8)
    inv = 1 - m                                         # fill holes
    n2, lab2, _, _ = cv2.connectedComponentsWithStats(inv, 4)
    border = np.unique(np.concatenate([lab2[0], lab2[-1], lab2[:, 0], lab2[:, -1]]))
    holes = ~np.isin(lab2, border) & (inv > 0)
    m[holes] = 1
    return m, bool(m.mean() >= min_frac)


def otsu_lesion(img):
    """Classical fallback lesion segmentation (only used if no GT masks are available)."""
    b = cv2.GaussianBlur(img[..., 2], (11, 11), 0)
    _, m = cv2.threshold(b, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return postprocess_lesion(m.astype(np.float32))


def detect_artifacts(img, L):
    """Rule-based artifact masks on a 256x256 RGB uint8 image.
    L: uint8 lesion mask. Artifact pixels are never inside the (dilated) lesion.
    Returns uint8 bitfield (DC|TS|SP|INK)."""
    H, W = L.shape
    area = H * W
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    Hh, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    Ld = cv2.dilate(L.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))) > 0

    # (1) dark corners / vignette: dark regions connected to the image border
    dark = cv2.morphologyEx((V < 50).astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(dark, 8)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    # must be a large blob (area) and not a thin line touching the border (hair): mean thickness >= 8 px
    keep = [b for b in border if b != 0 and st[b, cv2.CC_STAT_AREA] >= 0.002 * area and
            st[b, cv2.CC_STAT_AREA] / max(np.hypot(st[b, cv2.CC_STAT_WIDTH], st[b, cv2.CC_STAT_HEIGHT]), 1) >= 8]
    dc = np.isin(lab, keep) & ~Ld

    # (2) thin dark structures (hair, ruler ticks/lines): black-hat + elongation filter
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    bh = cv2.morphologyEx(cv2.GaussianBlur(gray, (3, 3), 0), cv2.MORPH_BLACKHAT, k)
    ts = (bh >= 18).astype(np.uint8)
    ts[cv2.dilate(dc.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0] = 0
    ts[:4, :] = 0; ts[-4:, :] = 0; ts[:, :4] = 0; ts[:, -4:] = 0     # v2: no frame-edge strips
    n, lab, st, _ = cv2.connectedComponentsWithStats(ts, 8)
    a = st[:, cv2.CC_STAT_AREA].astype(np.float64)
    diag = np.hypot(st[:, cv2.CC_STAT_WIDTH], st[:, cv2.CC_STAT_HEIGHT])
    good = (a >= 15) & (diag >= 15) & (a / np.maximum(diag, 1) <= 6.0)
    good[0] = False
    ts = good[lab] & ~Ld

    # (3) specular highlights / gel bubbles: small bright unsaturated blobs that are much brighter
    #     than their surround (v2: white top-hat keeps only bright peaks smaller than 11 px -> large
    #     whitish skin areas are rejected; bubble rims and glints are kept)
    tophat = cv2.morphologyEx(V, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))
    sp = ((V >= 210) & (S <= 50) & (tophat >= 25)).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(sp, 8)
    a = st[:, cv2.CC_STAT_AREA]
    good = (a >= 3) & (a <= 0.01 * area)
    good[0] = False
    sp = (cv2.dilate(good[lab].astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & ~Ld

    # (4) ink markings / coloured patches: strongly saturated blue-cyan-violet regions outside the lesion
    #     (v2: hue 90-150 also covers blue/cyan stickers; S>=90 rejects bluish-tinted skin; pink skin
    #      (H>150) excluded; a 12-px band next to the vignette is excluded (chromatic fringe))
    ink = cv2.morphologyEx(((Hh >= 90) & (Hh <= 150) & (S >= 90) & (V >= 40) & (V <= 235)).astype(np.uint8),
                           cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    good = st[:, cv2.CC_STAT_AREA] >= 40
    good[0] = False
    ink = good[lab] & ~Ld & ~(cv2.dilate(dc.astype(np.uint8), np.ones((25, 25), np.uint8)) > 0)

    return (dc * BIT_DC | ts * BIT_TS | sp * BIT_SP | ink * BIT_INK).astype(np.uint8)


def artifact_flags(mask_bits):
    """image-level presence flags from a (256,256) bitfield."""
    return {k: float(((mask_bits & b) > 0).mean()) >= ART_MIN_FRAC[k] for k, b in ART_BITS.items()}


# ----------------------------------------------------------------------------- GPU augmentation
def gpu_augment(img_u8, msk_u8, mode, out=OUT, gen=None):
    """img_u8 (B,256,256,3) uint8, msk_u8 (B,256,256) uint8 (on device).
    mode: 'eval' | 'std' | 'robust'. Image and masks receive the SAME geometric transform.
    Returns normalized image (B,3,out,out), lesion mask, artifact mask (B,1,out,out) in [0,1]."""
    dev = img_u8.device
    B = img_u8.shape[0]
    x = img_u8.permute(0, 3, 1, 2).float() / 255.0
    m = msk_u8.long()
    mL = ((m & BIT_L) > 0).float()[:, None]
    mA = ((m & BIT_ART) > 0).float()[:, None]
    R = lambda *s: torch.rand(*s, device=dev, generator=gen)
    theta = torch.zeros(B, 2, 3, device=dev)
    if mode == "eval":
        theta[:, 0, 0] = 1; theta[:, 1, 1] = 1
    elif mode == "std":       # Resize(256) + RandomCrop(224) + HFlip
        s = out / IMG
        fx = torch.where(R(B) < 0.5, -1.0, 1.0)
        theta[:, 0, 0] = s * fx; theta[:, 1, 1] = s
        theta[:, 0, 2] = (R(B) * 2 - 1) * (1 - s); theta[:, 1, 2] = (R(B) * 2 - 1) * (1 - s)
    elif mode == "robust":    # RandomResizedCrop(scale .4-1, ratio 3/4-4/3) + H/V flips (+ colour jitter)
        area = 0.4 + 0.6 * R(B)
        logr = math.log(3 / 4) + (math.log(4 / 3) - math.log(3 / 4)) * R(B)
        r = torch.exp(logr)
        sx = torch.sqrt(area * r).clamp(max=1.0); sy = torch.sqrt(area / r).clamp(max=1.0)
        fx = torch.where(R(B) < 0.5, -1.0, 1.0); fy = torch.where(R(B) < 0.5, -1.0, 1.0)
        theta[:, 0, 0] = sx * fx; theta[:, 1, 1] = sy * fy
        theta[:, 0, 2] = (R(B) * 2 - 1) * (1 - sx); theta[:, 1, 2] = (R(B) * 2 - 1) * (1 - sy)
    else:
        raise ValueError(mode)
    grid = F.affine_grid(theta, (B, 3, out, out), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    mm = F.grid_sample(torch.cat([mL, mA], 1), grid, mode="bilinear", padding_mode="border", align_corners=False)
    mL, mA = mm[:, :1], mm[:, 1:]
    if mode == "robust":      # ColorJitter(brightness=.2, contrast=.2, saturation=.2)
        b = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        x = (x * b).clamp(0, 1)
        c = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        g = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).mean((1, 2)).view(B, 1, 1, 1)
        x = ((x - g) * c + g).clamp(0, 1)
        s = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        gy = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2])[:, None]
        x = (gy + (x - gy) * s).clamp(0, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std, mL, mA


# ----------------------------------------------------------------------------- models
def densenet_features():
    from torchvision.models import densenet121, DenseNet121_Weights
    return densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1).features


class DenseCNN(nn.Module):
    """Black-box reference: DenseNet-121 + GAP + linear."""
    kind = "cnn"
    def __init__(self, n_classes=2, pretrained=True):
        super().__init__()
        self.features = densenet_features() if pretrained else _tiny_features()
        self.head = nn.Linear(1024, n_classes)
    def forward(self, x):
        f = F.relu(self.features(x))
        return {"logits": self.head(f.mean((2, 3))), "fmap": f}


class ProtoPNet(nn.Module):
    """ProtoPNet (Chen et al., NeurIPS 2019) with a DenseNet-121 backbone."""
    kind = "proto"
    def __init__(self, n_classes=2, ppc=10, D=128, pretrained=True):
        super().__init__()
        self.features = densenet_features() if pretrained else _tiny_features()
        self.add_on = nn.Sequential(nn.Conv2d(1024, D, 1), nn.ReLU(), nn.Conv2d(D, D, 1), nn.Sigmoid())
        P = n_classes * ppc
        self.prototypes = nn.Parameter(torch.rand(P, D, 1, 1))
        self.register_buffer("proto_class", torch.arange(P) // ppc)
        ident = F.one_hot(torch.arange(P) // ppc, n_classes).float().T          # (C,P)
        self.register_buffer("class_identity", ident)
        self.last = nn.Linear(P, n_classes, bias=False)
        with torch.no_grad():
            self.last.weight.copy_(1.0 * ident - 0.5 * (1 - ident))
        self.n_classes, self.P, self.D = n_classes, P, D

    def latent(self, x):
        f = F.relu(self.features(x))
        with torch.autocast("cuda", enabled=False):
            return self.add_on(f.float())

    def distances(self, z):
        x2 = (z ** 2).sum(1, keepdim=True)
        p2 = (self.prototypes ** 2).sum((1, 2, 3)).view(1, -1, 1, 1)
        xp = F.conv2d(z, self.prototypes)
        return F.relu(x2 + p2 - 2 * xp)

    def forward(self, x):
        z = self.latent(x)
        with torch.autocast("cuda", enabled=False):
            d = self.distances(z)
            sim = torch.log((d + 1) / (d + 1e-4))
            act = sim.amax((2, 3))
            logits = self.last(act)
        return {"logits": logits, "sim": sim, "dist": d, "min_d": d.amin((2, 3)), "act": act, "z": z}


def _tiny_features():  # CPU unit tests only
    return nn.Sequential(nn.Conv2d(3, 1024, 32, stride=32))


def build_model(cfg, pretrained=True):
    if cfg["model"] == "cnn":
        return DenseCNN(pretrained=pretrained)
    return ProtoPNet(ppc=cfg.get("ppc", 10), D=cfg.get("D", 128), pretrained=pretrained)


# ----------------------------------------------------------------------------- losses
def proto_base_losses(out, y, model, class_w):
    ce = F.cross_entropy(out["logits"], y, weight=class_w)
    md = out["min_d"]
    own = model.proto_class[None, :] == y[:, None]
    big = torch.finfo(md.dtype).max / 4
    clst = md.masked_fill(~own, big).amin(1).mean()
    sep = md.masked_fill(own, big).amin(1).mean()
    return ce, clst, sep


def artifact_lesion_losses(out, y, model, cL, cA, valid, tau=0.5):
    """Proposed losses.  sim: (B,P,h,w); cL/cA: lesion/artifact fraction per latent cell (B,1,h,w).
    L_art : activation-weighted spatial-attention mass that every prototype puts on artifact cells.
    L_les : -log attention mass that own-class prototypes put on lesion cells."""
    sim = out["sim"]
    B, P, h, w = sim.shape
    att = F.softmax(sim.flatten(2) / tau, dim=2)                         # (B,P,hw)
    wt = sim.flatten(2).amax(2).detach()
    wt = wt / wt.sum(1, keepdim=True).clamp_min(1e-8)
    a_art = (att * cA.flatten(2)).sum(2)                                  # (B,P)
    l_art_i = (wt * a_art).sum(1)
    own = (model.proto_class[None, :] == y[:, None]).float()
    w_own = wt * own
    w_own = w_own / w_own.sum(1, keepdim=True).clamp_min(1e-8)
    a_les = (att * cL.flatten(2)).sum(2)
    l_les_i = (w_own * -torch.log(a_les + 1e-4)).sum(1)
    has_les = valid & (cL.flatten(1).sum(1) >= 0.5)
    zero = sim.sum() * 0
    l_art = l_art_i[valid].mean() if valid.any() else zero
    l_les = l_les_i[has_les].mean() if has_les.any() else zero
    return l_art, l_les, a_art.detach(), a_les.detach()


def fine_annotation_loss(out, y, model, cL, valid):
    """IAIA-BL style fine-annotation loss (Barnett et al., 2021) adapted to lesion masks:
    own-class prototypes are penalised for activating outside the lesion, other-class
    prototypes for activating anywhere."""
    sim = out["sim"]
    own = model.proto_class[None, :] == y[:, None]
    t_own = (sim * (1 - cL)).flatten(2).norm(dim=2)
    t_oth = sim.flatten(2).norm(dim=2)
    per = (t_own * own).sum(1) + (t_oth * ~own).sum(1)
    return per[valid].mean() if valid.any() else sim.sum() * 0


def cell_fractions(mL, mA, hw):
    return F.adaptive_avg_pool2d(mL, hw), F.adaptive_avg_pool2d(mA, hw)


# ----------------------------------------------------------------------------- data
class ArrDS(torch.utils.data.Dataset):
    def __init__(self, img_path, msk_path, labels, valid, idx):
        self.img_path, self.msk_path = img_path, msk_path
        self.labels = np.asarray(labels).astype(np.int64)
        self.valid = np.asarray(valid).astype(bool)
        self.idx = np.asarray(idx)
        self.imgs = self.msks = None
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        if self.imgs is None:
            self.imgs = np.load(self.img_path, mmap_mode="r")
            self.msks = np.load(self.msk_path, mmap_mode="r")
        i = int(self.idx[k])
        return (torch.from_numpy(np.ascontiguousarray(self.imgs[i])),
                torch.from_numpy(np.ascontiguousarray(self.msks[i])),
                int(self.labels[i]), bool(self.valid[i]), i)


def make_loader(ds, bs, shuffle, seed=0, workers=4, drop_last=False):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=workers,
                                       pin_memory=torch.cuda.is_available(), drop_last=drop_last,
                                       persistent_workers=workers > 0)


# ----------------------------------------------------------------------------- inference
@torch.no_grad()
def predict(model, loader, device, keep_maps=False):
    model.eval()
    probs, ys, ids, maps, acts = [], [], [], [], []
    for img, msk, y, v, i in loader:
        x, _, _ = gpu_augment(img.to(device, non_blocking=True), msk.to(device, non_blocking=True), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model(x.contiguous(memory_format=torch.channels_last))
        probs.append(F.softmax(out["logits"].float(), 1)[:, 1].cpu()); ys.append(y); ids.append(i)
        if keep_maps and "sim" in out:
            maps.append(out["sim"].half().cpu()); acts.append(out["act"].cpu())
    r = {"prob": torch.cat(probs).numpy(), "y": torch.cat(ys).numpy(), "idx": torch.cat(ids).numpy()}
    if keep_maps and maps:
        r["sim"] = torch.cat(maps).numpy(); r["act"] = torch.cat(acts).numpy()
    return r


def auc_safe(y, p):
    from sklearn.metrics import roc_auc_score
    try:
        return float(roc_auc_score(y, p))
    except Exception:
        return float("nan")


# ----------------------------------------------------------------------------- push
@torch.no_grad()
def push_prototypes(model, loader, device, constrained, les_min=0.5, art_max=0.05):
    """Project every prototype onto its nearest latent patch of a training image of its class.
    constrained=True: candidate patches must lie on the lesion (lesion fraction >= les_min)
    and be artifact-free (artifact fraction <= art_max)."""
    model.eval()
    P, D = model.P, model.D
    best_d = torch.full((P,), float("inf"), device=device)
    best_z = model.prototypes.data.view(P, D).clone()
    info = [dict(proto=j, cls=int(model.proto_class[j]), img=-1, h=-1, w=-1, dist=float("inf"),
                 les_frac=float("nan"), art_frac=float("nan")) for j in range(P)]
    for img, msk, y, v, ii in loader:
        img, msk, y, v = img.to(device), msk.to(device), y.to(device), v.to(device)
        x, mL, mA = gpu_augment(img, msk, "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            z = model.latent(x.contiguous(memory_format=torch.channels_last))
        d = model.distances(z.float())                                    # (B,P,h,w)
        B, _, h, w = d.shape
        cL, cA = cell_fractions(mL, mA, (h, w))
        ok = torch.ones(B, 1, h, w, dtype=torch.bool, device=device)
        if constrained:
            ok = (cL >= les_min) & (cA <= art_max) & v.view(B, 1, 1, 1)
        own = model.proto_class.view(1, P, 1, 1) == y.view(B, 1, 1, 1)
        dd = d.masked_fill(~(own & ok), float("inf")).permute(1, 0, 2, 3).reshape(P, -1)
        val, arg = dd.min(1)
        upd = val < best_d
        for j in torch.nonzero(upd).flatten().tolist():
            a = int(arg[j]); b, rem = divmod(a, h * w); hh, ww = divmod(rem, w)
            best_d[j] = val[j]; best_z[j] = z[b, :, hh, ww].float()
            info[j].update(img=int(ii[b]), h=hh, w=ww, dist=float(val[j]),
                           les_frac=float(cL[b, 0, hh, ww]), art_frac=float(cA[b, 0, hh, ww]))
    model.prototypes.data.copy_(best_z.view(P, D, 1, 1))
    return pd.DataFrame(info)


# ----------------------------------------------------------------------------- training
DEFAULT = dict(model="proto", aug="robust", lam_clst=0.8, lam_sep=0.08, lam_l1=1e-4,
               lam_art=0.0, lam_les=0.0, lam_fa=0.0, tau=0.5, push="standard",
               ep_warm=5, ep_joint=15, ep_ll=5, bs=64, lr_feat=1e-4, lr_addon=3e-3, lr_proto=3e-3,
               lr_head=1e-3, lr_ll=1e-3, wd=1e-3, seed=0)

CONFIGS = {
    "CNN":         dict(model="cnn"),
    "PPN-std":     dict(aug="std"),
    "PPN":         dict(),
    "PPN-FA":      dict(lam_fa=1e-3),
    "Ours":        dict(lam_art=1.0, lam_les=1.0, push="constrained"),
    "Ours-noLes":  dict(lam_art=1.0, lam_les=0.0, push="constrained"),
    "Ours-noArt":  dict(lam_art=0.0, lam_les=1.0, push="constrained"),
    "Ours-stdPush": dict(lam_art=1.0, lam_les=1.0, push="standard"),
}


def full_cfg(name, seed):
    c = dict(DEFAULT); c.update(CONFIGS[name]); c["name"] = name; c["seed"] = seed
    return c


def _set_trainable(model, stage):
    for p in model.parameters(): p.requires_grad = False
    if model.kind == "cnn":
        for p in model.head.parameters(): p.requires_grad = True
        if stage == "joint":
            for p in model.features.parameters(): p.requires_grad = True
        return
    if stage in ("warm", "joint"):
        for p in model.add_on.parameters(): p.requires_grad = True
        model.prototypes.requires_grad = True
        if stage == "joint":
            for p in model.features.parameters(): p.requires_grad = True
    elif stage == "last":
        for p in model.last.parameters(): p.requires_grad = True


def train_run(cfg, prep, outdir, device, smoke=False, workers=4, log=print):
    """prep: dict(img_path, msk_path, meta DataFrame with split,label,lesion_valid)."""
    os.makedirs(outdir, exist_ok=True)
    set_seed(cfg["seed"])
    torch.backends.cudnn.benchmark = True
    meta = prep["meta"]
    tr = np.where(meta.split.values == "train")[0]
    va = np.where(meta.split.values == "val")[0]
    if smoke:
        rs = np.random.RandomState(0)
        tr, va = rs.choice(tr, min(512, len(tr)), replace=False), rs.choice(va, min(256, len(va)), replace=False)
        cfg = dict(cfg, ep_warm=1, ep_joint=1, ep_ll=1)
    labels, valid = meta.label.values, meta.lesion_valid.values
    mk = lambda idx: ArrDS(prep["img_path"], prep["msk_path"], labels, valid, idx)
    tr_loader = make_loader(mk(tr), cfg["bs"], True, cfg["seed"], workers, drop_last=True)
    tr_eval_loader = make_loader(mk(tr), 128, False, 0, workers)
    va_loader = make_loader(mk(va), 128, False, 0, workers)
    n1 = labels[tr].sum(); n0 = len(tr) - n1
    class_w = torch.tensor([len(tr) / (2 * n0), len(tr) / (2 * n1)], dtype=torch.float32, device=device)

    model = build_model(cfg, pretrained=prep.get("pretrained", True)).to(device).to(memory_format=torch.channels_last)
    is_proto = model.kind == "proto"
    if is_proto:
        opt_warm = torch.optim.Adam([{"params": model.add_on.parameters(), "lr": cfg["lr_addon"], "weight_decay": cfg["wd"]},
                                     {"params": [model.prototypes], "lr": cfg["lr_proto"]}])
        opt_joint = torch.optim.Adam([{"params": model.features.parameters(), "lr": cfg["lr_feat"], "weight_decay": cfg["wd"]},
                                      {"params": model.add_on.parameters(), "lr": cfg["lr_addon"], "weight_decay": cfg["wd"]},
                                      {"params": [model.prototypes], "lr": cfg["lr_proto"]}])
    else:
        opt_warm = torch.optim.Adam(model.head.parameters(), lr=cfg["lr_head"])
        opt_joint = torch.optim.Adam([{"params": model.features.parameters(), "lr": cfg["lr_feat"], "weight_decay": cfg["wd"]},
                                      {"params": model.head.parameters(), "lr": cfg["lr_head"]}])
    sched = torch.optim.lr_scheduler.StepLR(opt_joint, step_size=5, gamma=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    hist, best = [], dict(auc=-1, state=None, epoch=-1)
    E = cfg["ep_warm"] + cfg["ep_joint"]
    t_start = time.time()

    def one_epoch(opt, stage):
        # last-layer stage: everything below the last layer is frozen INCLUDING BatchNorm statistics,
        # so the latent space (and hence the pushed prototypes = real training patches) stays exact.
        model.train(stage != "last")
        agg = {}
        for img, msk, y, v, ii in tr_loader:
            img, msk = img.to(device, non_blocking=True), msk.to(device, non_blocking=True)
            y, v = y.to(device), v.to(device)
            x, mL, mA = gpu_augment(img, msk, cfg["aug"])
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                out = model(x.contiguous(memory_format=torch.channels_last))
            parts = {}
            if is_proto:
                ce, clst, sep = proto_base_losses(out, y, model, class_w)
                parts.update(ce=ce, clst=clst, sep=sep)
                if stage == "last":
                    l1 = (model.last.weight * (1 - model.class_identity)).norm(p=1)
                    loss = ce + cfg["lam_l1"] * l1
                else:
                    loss = ce + cfg["lam_clst"] * clst - cfg["lam_sep"] * sep
                    if cfg["lam_art"] > 0 or cfg["lam_les"] > 0:
                        cL, cA = cell_fractions(mL, mA, out["sim"].shape[2:])
                        la, ll, _, _ = artifact_lesion_losses(out, y, model, cL, cA, v, cfg["tau"])
                        loss = loss + cfg["lam_art"] * la + cfg["lam_les"] * ll
                        parts.update(art=la, les=ll)
                    if cfg["lam_fa"] > 0:
                        cL, _ = cell_fractions(mL, mA, out["sim"].shape[2:])
                        fa = fine_annotation_loss(out, y, model, cL, v)
                        loss = loss + cfg["lam_fa"] * fa
                        parts.update(fa=fa)
            else:
                loss = F.cross_entropy(out["logits"].float(), y, weight=class_w)
                parts.update(ce=loss)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            for k, t in parts.items():
                agg[k] = agg.get(k, 0.0) + float(t.detach())
            agg["n"] = agg.get("n", 0) + 1
        n = agg.pop("n", 1)
        return {k: round(t / n, 4) for k, t in agg.items()}

    for ep in range(1, E + 1):
        stage = "warm" if ep <= cfg["ep_warm"] else "joint"
        _set_trainable(model, stage)
        t0 = time.time()
        tl = one_epoch(opt_warm if stage == "warm" else opt_joint, stage)
        if stage == "joint": sched.step()
        pv = predict(model, va_loader, device)
        auc = auc_safe(pv["y"], pv["prob"])
        rec = dict(epoch=ep, stage=stage, val_auc=round(auc, 4), sec=round(time.time() - t0, 1), **tl)
        hist.append(rec); log(f"[{cfg['name']} s{cfg['seed']}] {rec}")
        if not is_proto and auc > best["auc"]:
            best = dict(auc=auc, state=copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()}), epoch=ep)

    push_info = None
    if is_proto:
        push_info = push_prototypes(model, tr_eval_loader, device, constrained=cfg["push"] == "constrained")
        pv = predict(model, va_loader, device)
        auc = auc_safe(pv["y"], pv["prob"])
        hist.append(dict(epoch=E, stage="after_push", val_auc=round(auc, 4)))
        log(f"[{cfg['name']} s{cfg['seed']}] after push val_auc={auc:.4f} | protos on artifact cells(>0.5): "
            f"{int((push_info.art_frac > 0.5).sum())}/{len(push_info)} | on lesion cells(>=0.5): "
            f"{int((push_info.les_frac >= 0.5).sum())}/{len(push_info)}")
        best = dict(auc=auc, state=copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()}), epoch=E)
        _set_trainable(model, "last")
        opt_ll = torch.optim.Adam(model.last.parameters(), lr=cfg["lr_ll"])
        for k in range(1, cfg["ep_ll"] + 1):
            t0 = time.time()
            tl = one_epoch(opt_ll, "last")
            pv = predict(model, va_loader, device)
            auc = auc_safe(pv["y"], pv["prob"])
            rec = dict(epoch=E + k, stage="last", val_auc=round(auc, 4), sec=round(time.time() - t0, 1), **tl)
            hist.append(rec); log(f"[{cfg['name']} s{cfg['seed']}] {rec}")
            if auc > best["auc"]:
                best = dict(auc=auc, state=copy.deepcopy({k2: t.cpu() for k2, t in model.state_dict().items()}), epoch=E + k)
        push_info.to_csv(os.path.join(outdir, "push_info.csv"), index=False)

    if best["state"] is None:   # e.g. undefined val AUC: never save an empty checkpoint
        best = dict(auc=float("nan"), state={k: t.cpu() for k, t in model.state_dict().items()}, epoch=len(hist))
    torch.save(best["state"], os.path.join(outdir, "model.pt"))
    json.dump(dict(cfg=cfg, hist=hist, best_epoch=best["epoch"], best_val_auc=best["auc"],
                   minutes=round((time.time() - t_start) / 60, 1)), open(os.path.join(outdir, "log.json"), "w"), indent=1)
    log(f"[{cfg['name']} s{cfg['seed']}] DONE best_val_auc={best['auc']:.4f} at epoch {best['epoch']} "
        f"({(time.time() - t_start) / 60:.1f} min)")
    return best["auc"]


def find_prep_dirs(files=None):
    """Locate NB00 output (dir with meta.csv + images.npy) and NB00b output (dir with meta.csv + masks.npy +
    dup_candidates.csv), independent of how Kaggle mounts notebook outputs. Prints a diagnosis if missing."""
    files = files if files is not None else walk_files("/kaggle/input") + walk_files("/kaggle/working")
    by_dir = {}
    for f in files:
        by_dir.setdefault(os.path.dirname(f), set()).add(os.path.basename(f))
    p1 = sorted(d for d, fs in by_dir.items() if {"meta.csv", "images.npy"} <= fs)
    p2 = sorted(d for d, fs in by_dir.items() if {"meta.csv", "masks.npy", "dup_candidates.csv"} <= fs)
    if not p1:
        print("!!! NB00 output NOT found under /kaggle/input. What IS mounted (first 3 levels):")
        for d in sorted({"/".join(f.split("/")[:6]) for f in files})[:40]:
            print("   ", d)
        print("   any meta.csv:", [f for f in files if f.endswith("meta.csv")][:10])
    return (p1[0] if p1 else None), (p2[0] if p2 else None)


In [ ]:
%%writefile evalkit.py
"""
evalkit.py -- evaluation toolkit for the paper (NB02).
Classification metrics + DeLong + bootstrap, convex last-layer refit, prototype attribution
(lesion / artifact / background hits), Grad-CAM for the CNN, controlled counterfactual
artifact interventions (lesion pixels provably unchanged) and explanation-stability metrics.
"""
import os, json, math, numpy as np, pandas as pd, cv2, torch
import torch.nn.functional as F
from scipy.stats import norm, wilcoxon
from sklearn.metrics import roc_auc_score
import artproto as ap

OUT = ap.OUT  # 224


# ----------------------------------------------------------------------------- runs
def find_runs(files):
    by = {}
    for f in files:
        by.setdefault(os.path.dirname(f), set()).add(os.path.basename(f))
    runs = {}
    for d, fs in by.items():
        name = os.path.basename(d)
        if {"model.pt", "log.json"} <= fs and not name.endswith("_smoke"):
            if name not in runs or d.startswith("/kaggle/working"):
                runs[name] = d
    return dict(sorted(runs.items()))


def load_run(d, device):
    L = json.load(open(f"{d}/log.json"))
    cfg = L["cfg"]
    sd = torch.load(f"{d}/model.pt", map_location="cpu")
    m = ap.build_model(cfg, pretrained=False)
    if "features.conv0.weight" in sd:          # real DenseNet-121 checkpoints (no weight download needed)
        import torchvision
        m.features = torchvision.models.densenet121(weights=None).features
    m.load_state_dict(sd, strict=True)
    m = m.to(device).eval().to(memory_format=torch.channels_last)
    return m, cfg, L


# ----------------------------------------------------------------------------- forward passes
def _batches(n, bs):
    for s in range(0, n, bs):
        yield s, min(n, s + bs)


@torch.no_grad()
def forward(model, imgs, msks, idx, device, bs=128, keep_maps=False):
    """imgs/msks: arrays or memmaps (N,256,256,3)/(N,256,256); idx: positions to use.
    Returns dict prob (n,), act (n,P) and optionally sim (n,P,7,7) float16."""
    model.eval()
    probs, acts, sims = [], [], []
    for s, e in _batches(len(idx), bs):
        ii = idx[s:e]
        x, _, _ = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(imgs[ii])).to(device),
                                 torch.from_numpy(np.ascontiguousarray(msks[ii])).to(device), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model(x.contiguous(memory_format=torch.channels_last))
        probs.append(F.softmax(out["logits"].float(), 1)[:, 1].cpu())
        if "act" in out:
            acts.append(out["act"].float().cpu())
            if keep_maps:
                sims.append(out["sim"].half().cpu())
    r = {"prob": torch.cat(probs).numpy()}
    if acts:
        r["act"] = torch.cat(acts).numpy()
        if keep_maps:
            r["sim"] = torch.cat(sims).numpy()
    return r


@torch.no_grad()
def gradcam(model, imgs, msks, idx, device, bs=128):
    """Class-activation map of the melanoma logit. For a global-average-pooling + linear head,
    d(logit_mel)/d(f) = w_mel/(h*w) at every position, so Grad-CAM equals CAM (Zhou et al. 2016)
    up to a positive constant; this formulation needs no backward pass (low memory)."""
    model.eval()
    w = model.head.weight[1].float()
    cams = []
    for s, e in _batches(len(idx), bs):
        ii = idx[s:e]
        x, _, _ = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(imgs[ii])).to(device),
                                 torch.from_numpy(np.ascontiguousarray(msks[ii])).to(device), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            f = F.relu(model.features(x.contiguous(memory_format=torch.channels_last)))
        cams.append(F.relu(torch.einsum("bchw,c->bhw", f.float(), w)).half().cpu())
    return torch.cat(cams).numpy()


# ----------------------------------------------------------------------------- convex last layer
def refit_last_layer(model, acts, y, class_w, l1=1e-4, steps=3000, lr=0.02):
    """Optimise the ProtoPNet last layer (linear map on prototype activations) to convergence:
    weighted CE + l1 on negative-class connections (as in Chen et al.). Deterministic, no val peeking."""
    dev = model.last.weight.device
    W = model.last.weight.detach().clone().float().requires_grad_(True)
    A = torch.as_tensor(acts, dtype=torch.float32, device=dev)
    Y = torch.as_tensor(y, dtype=torch.long, device=dev)
    cw = torch.as_tensor(class_w, dtype=torch.float32, device=dev)
    neg = (1 - model.class_identity).float()
    opt = torch.optim.Adam([W], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for _ in range(steps):
        loss = F.cross_entropy(A @ W.T, Y, weight=cw) + l1 * (W * neg).abs().sum()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return W.detach(), float(loss.detach())


def probs_from_act(act, W):
    return torch.softmax(torch.as_tensor(act) @ torch.as_tensor(W).cpu().T, 1)[:, 1].numpy()


# ----------------------------------------------------------------------------- metrics
def youden_threshold(y, p):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y, p)
    k = int(np.argmax(tpr - fpr))
    return float(min(thr[k], 1.0))


def bin_metrics(y, p, thr):
    y = np.asarray(y).astype(int); pr = (np.asarray(p) >= thr).astype(int)
    tp = int(((pr == 1) & (y == 1)).sum()); tn = int(((pr == 0) & (y == 0)).sum())
    fp = int(((pr == 1) & (y == 0)).sum()); fn = int(((pr == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); ppv = tp / max(tp + fp, 1)
    auc = roc_auc_score(y, p) if 0 < y.sum() < len(y) else float("nan")
    return dict(AUC=auc, BAcc=(sens + spec) / 2, Sens=sens, Spec=spec, PPV=ppv,
                F1=2 * ppv * sens / max(ppv + sens, 1e-9), TP=tp, FN=fn, FP=fp, TN=tn)


def _midrank(x):
    J = np.argsort(x, kind="mergesort"); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1); i = j
    T2 = np.empty(N); T2[J] = T + 1
    return T2


def delong(y, p1, p2):
    """Paired DeLong test (Sun & Xu 2014). Returns auc1, auc2, z, two-sided p."""
    y = np.asarray(y).astype(int)
    order = np.r_[np.where(y == 1)[0], np.where(y == 0)[0]]
    m = int(y.sum()); n = len(y) - m
    P = np.vstack([p1, p2])[:, order].astype(np.float64)
    tx = np.array([_midrank(r[:m]) for r in P]); ty = np.array([_midrank(r[m:]) for r in P])
    tz = np.array([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / math.sqrt(max(var, 1e-12))
    return float(aucs[0]), float(aucs[1]), float(z), float(2 * norm.sf(abs(z)))


def bootstrap_auc_ci(y, p, groups=None, B=1000, seed=0):
    rs = np.random.RandomState(seed); y = np.asarray(y); p = np.asarray(p)
    if groups is None:
        groups = np.arange(len(y))
    ug, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == g)[0] for g in range(len(ug))]
    vals = []
    for _ in range(B):
        pick = rs.randint(0, len(ug), len(ug))
        ii = np.concatenate([members[g] for g in pick])
        if 0 < y[ii].sum() < len(ii):
            vals.append(roc_auc_score(y[ii], p[ii]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# ----------------------------------------------------------------------------- masks at network resolution
def masks224(msk_u8, device):
    """bit-field masks (b,256,256) -> dict of bool tensors (b,224,224) with the SAME resampling as the
    evaluation transform (bilinear, align_corners=False) and a 0.5 threshold."""
    m = torch.as_tensor(np.ascontiguousarray(msk_u8)).to(device).long()
    chans = {"L": (m & ap.BIT_L) > 0, "A": (m & ap.BIT_ART) > 0}
    for k, b in ap.ART_BITS.items():
        chans[k] = (m & b) > 0
    st = torch.stack([v.float() for v in chans.values()], 1)
    st = F.interpolate(st, size=(OUT, OUT), mode="bilinear", align_corners=False) >= 0.5
    return {k: st[:, i] for i, k in enumerate(chans)}


def hot_regions(maps7, size=OUT, q=0.95):
    """maps7 (b,k,7,7) float -> boolean top-(1-q) regions (b,k,size,size) after bilinear upsampling."""
    b, k = maps7.shape[:2]
    up = F.interpolate(maps7.float(), size=(size, size), mode="bilinear", align_corners=False)
    flat = up.reshape(b, k, -1)
    thr = torch.quantile(flat, q, dim=2, keepdim=True)
    return (flat >= thr).reshape(b, k, size, size), up


def region_stats(H, up, M):
    """H (b,k,S,S) bool hot regions; up the maps; M dict of (b,S,S) bool masks.
    Returns per-image means over the k maps: hit fraction of each mask + pointing (argmax in mask)."""
    hs = H.float().sum((2, 3)).clamp_min(1)
    arg = up.reshape(*up.shape[:2], -1).argmax(2)                     # (b,k)
    out = {}
    for name, m in M.items():
        mm = m[:, None].float()
        out[f"hit_{name}"] = ((H.float() * mm).sum((2, 3)) / hs).mean(1)
        out[f"point_{name}"] = torch.gather(m.reshape(m.shape[0], -1).float(), 1, arg).mean(1)
    return {k: v.cpu().numpy() for k, v in out.items()}


def attribution(maps7, acts, msk_u8, device, topk=3, q=0.95, bs=64):
    """Per-image attribution statistics for prototype models (top-k most activated prototypes) or the
    CNN (acts=None: maps7 is the (n,7,7) Grad-CAM)."""
    rows = {}
    n = len(maps7)
    for s, e in _batches(n, bs):
        mp = torch.as_tensor(maps7[s:e]).to(device).float()
        if acts is None:
            sel = mp[:, None]
        else:
            top = torch.as_tensor(np.argsort(-acts[s:e], 1)[:, :topk].copy()).to(device)
            sel = torch.gather(mp, 1, top[:, :, None, None].expand(-1, -1, mp.shape[2], mp.shape[3]))
        H, up = hot_regions(sel, q=q)
        M = masks224(msk_u8[s:e], device)
        M["B"] = ~M["L"] & ~M["A"]
        st = region_stats(H, up, M)
        for k, v in st.items():
            rows.setdefault(k, []).append(v)
        for k in ["L", "A"] + list(ap.ART_BITS):
            rows.setdefault(f"area_{k}", []).append(M[k].float().mean((1, 2)).cpu().numpy())
    return pd.DataFrame({k: np.concatenate(v) for k, v in rows.items()})


# ----------------------------------------------------------------------------- counterfactual interventions
def _allowed(L, margin=4):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1))
    return cv2.dilate(L.astype(np.uint8), k) == 0


def _compose(img, layer, allowed):
    changed = (layer != img).any(-1) & allowed
    out = img.copy(); out[changed] = layer[changed]
    return out, changed


def add_hair(img, L, rng):
    layer = img.copy()
    for _ in range(int(rng.integers(12, 30))):
        p0 = rng.uniform(0, 256, 2); a = rng.uniform(0, 2 * np.pi); ln = rng.uniform(60, 200)
        p2 = p0 + ln * np.array([np.cos(a), np.sin(a)]); p1 = (p0 + p2) / 2 + rng.normal(0, 25, 2)
        t = np.linspace(0, 1, 60)[:, None]
        pts = ((1 - t) ** 2 * p0 + 2 * (1 - t) * t * p1 + t ** 2 * p2).astype(np.int32)
        c = rng.uniform(15, 70)
        cv2.polylines(layer, [pts], False, (int(c * 1.15), int(c * 0.95), int(c * 0.8)),
                      int(rng.integers(1, 3)), cv2.LINE_AA)
    return _compose(img, layer, _allowed(L))


def add_ruler(img, L, rng):
    allowed = _allowed(L)
    for side in rng.permutation(["top", "bottom", "left", "right"]):
        layer = img.copy(); col = int(rng.uniform(35, 80)); ln = int(rng.uniform(150, 220))
        off = int(rng.uniform(10, 30)); s0 = (256 - ln) // 2
        canvas = np.zeros((256, 256), np.uint8)
        cv2.line(canvas, (s0, off), (s0 + ln, off), 255, 2, cv2.LINE_AA)
        for k, x in enumerate(range(s0, s0 + ln + 1, 7)):
            cv2.line(canvas, (x, off), (x, off + (12 if k % 5 == 0 else 6)), 255, 1, cv2.LINE_AA)
        if side == "bottom": canvas = canvas[::-1]
        elif side == "left": canvas = canvas.T.copy()
        elif side == "right": canvas = canvas.T[:, ::-1].copy()
        drawn = canvas > 0
        if (drawn & ~allowed).sum() > 0.02 * drawn.sum():
            continue                                    # would touch the lesion margin -> try another side
        a = (canvas.astype(np.float32) / 255)[..., None]
        layer = (img * (1 - a) + col * a).astype(np.uint8)
        return _compose(img, layer, allowed)
    return None, None


def add_dark_corners(img, L, rng):
    yy, xx = np.mgrid[:256, :256]
    r = np.hypot(xx - 127.5, yy - 127.5) / (127.5 * np.sqrt(2))
    r0 = rng.uniform(0.72, 0.85)
    a = np.clip((r - r0) / 0.06, 0, 1)[..., None]
    layer = (img * (1 - a) + rng.uniform(0, 15) * a).astype(np.uint8)
    return _compose(img, layer, _allowed(L))


def add_ink(img, L, rng):
    ys, xs = np.where(L > 0)
    if len(ys) == 0:
        return None, None
    cy, cx = ys.mean(), xs.mean()
    ry = (ys.max() - ys.min()) / 2 + rng.uniform(12, 25); rx = (xs.max() - xs.min()) / 2 + rng.uniform(12, 25)
    layer = img.copy(); col = (int(rng.uniform(100, 135)), int(rng.uniform(30, 55)), int(rng.uniform(130, 170)))
    start = rng.uniform(0, 360)
    for k in range(int(rng.integers(2, 5))):
        a0 = start + k * 90 + rng.uniform(-10, 10)
        cv2.ellipse(layer, (int(cx), int(cy)), (int(rx), int(ry)), 0, a0, a0 + rng.uniform(35, 70), col, 3, cv2.LINE_AA)
    return _compose(img, layer, _allowed(L))


def remove_by_inpaint(img, L, mask):
    m = (cv2.dilate(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & _allowed(L, 2)
    if m.sum() == 0:
        return None, None
    inp = cv2.inpaint(img, m.astype(np.uint8), 3, cv2.INPAINT_TELEA)
    out = img.copy(); out[m] = inp[m]
    return out, m


def null_inpaint(img, L, art_mask, rng):
    """Control: inpaint a region of the same shape as the artifact mask, shifted onto clean skin."""
    for _ in range(10):
        dy, dx = rng.integers(-100, 100, 2)
        sh = np.roll(np.roll(art_mask, dy, 0), dx, 1) & ~art_mask
        if sh.sum() >= 0.5 * art_mask.sum():
            return remove_by_inpaint(img, L, sh)
    return None, None


INTERVENTIONS = ["add_hair", "add_ruler", "add_dark_corners", "add_ink", "remove_hair_ruler", "null_inpaint"]


def make_counterfactuals(IM, MS, idx, kind, seed=0):
    """Returns (cf images (n,256,256,3) uint8, changed masks (n,256,256) bool, ok flags) for test indices idx.
    Lesion pixels are verified to be bit-identical to the original."""
    n = len(idx)
    CF = np.zeros((n, 256, 256, 3), np.uint8); CH = np.zeros((n, 256, 256), bool); ok = np.zeros(n, bool)
    for k, i in enumerate(idx):
        img = np.array(IM[i]); m = np.array(MS[i]); L = (m & ap.BIT_L) > 0
        rng = np.random.default_rng([seed, int(i), INTERVENTIONS.index(kind)])
        if kind == "add_hair": out, ch = add_hair(img, L, rng)
        elif kind == "add_ruler": out, ch = add_ruler(img, L, rng)
        elif kind == "add_dark_corners": out, ch = add_dark_corners(img, L, rng)
        elif kind == "add_ink": out, ch = add_ink(img, L, rng)
        elif kind == "remove_hair_ruler":
            ts = (m & ap.BIT_TS) > 0
            out, ch = remove_by_inpaint(img, L, ts) if ts.mean() >= ap.ART_MIN_FRAC["TS"] else (None, None)
        elif kind == "null_inpaint":
            ts = (m & ap.BIT_TS) > 0
            out, ch = null_inpaint(img, L, ts | False, rng) if ts.mean() >= ap.ART_MIN_FRAC["TS"] else (None, None)
            if out is not None:
                ch = ch & ((m & ap.BIT_ART) == 0)
                out2 = img.copy(); out2[ch] = out[ch]; out = out2
        else:
            raise ValueError(kind)
        if out is None or ch.sum() == 0:
            CF[k] = img; continue
        assert np.array_equal(out[L], img[L]), f"lesion pixels changed for {kind} image {i}"
        CF[k] = out; CH[k] = ch; ok[k] = True
    return CF, CH, ok


def stability(sim0, act0, sim1, act1, changed256, msk_u8, device, topk=3, q=0.95, bs=64):
    """Explanation stability for prototype models: same top-k prototypes (chosen on the original image)
    before vs after the intervention."""
    rows = {}
    for s, e in _batches(len(sim0), bs):
        a0 = act0[s:e]; top = torch.as_tensor(np.argsort(-a0, 1)[:, :topk].copy()).to(device)
        g = lambda S: torch.gather(torch.as_tensor(S[s:e]).to(device).float(), 1,
                                   top[:, :, None, None].expand(-1, -1, S.shape[2], S.shape[3]))
        m0, m1 = g(sim0), g(sim1)
        H0, _ = hot_regions(m0, q=q); H1, _ = hot_regions(m1, q=q)
        M = masks224(msk_u8[s:e], device)
        ch = F.interpolate(torch.as_tensor(changed256[s:e]).to(device).float()[:, None], size=(OUT, OUT),
                           mode="bilinear", align_corners=False)[:, 0] >= 0.5
        inter = (H0 & H1).float().sum((2, 3)); uni = (H0 | H1).float().sum((2, 3)).clamp_min(1)
        L = M["L"][:, None].float()
        h1 = H1.float().sum((2, 3)).clamp_min(1); h0 = H0.float().sum((2, 3)).clamp_min(1)
        rows.setdefault("iou", []).append((inter / uni).mean(1).cpu().numpy())
        rows.setdefault("les_before", []).append(((H0.float() * L).sum((2, 3)) / h0).mean(1).cpu().numpy())
        rows.setdefault("les_after", []).append(((H1.float() * L).sum((2, 3)) / h1).mean(1).cpu().numpy())
        rows.setdefault("capture", []).append(((H1.float() * ch[:, None].float()).sum((2, 3)) / h1).mean(1).cpu().numpy())
        t1 = np.argsort(-act1[s:e], 1)[:, :topk]; t0 = np.argsort(-a0, 1)[:, :topk]
        rows.setdefault("top_jaccard", []).append(np.array([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(t0, t1)]))
        x0 = m0[:, 0].reshape(m0.shape[0], -1); x1 = m1[:, 0].reshape(m1.shape[0], -1)
        r0 = x0.argsort(1).argsort(1).float(); r1 = x1.argsort(1).argsort(1).float()
        r0 = r0 - r0.mean(1, keepdim=True); r1 = r1 - r1.mean(1, keepdim=True)
        rows.setdefault("spearman_top1", []).append(((r0 * r1).sum(1) / (r0.norm(dim=1) * r1.norm(dim=1)).clamp_min(1e-9)).cpu().numpy())
    return pd.DataFrame({k: np.concatenate(v) for k, v in rows.items()})


def paired_wilcoxon(a, b):
    d = np.asarray(a) - np.asarray(b)
    d = d[np.isfinite(d)]
    if len(d) < 10 or np.allclose(d, 0):
        return float("nan")
    return float(wilcoxon(d).pvalue)


In [ ]:
%%writefile fa_ext.py
"""Mask-loss formulations (as in NB06) plus the configuration Ours+Correia."""
import torch, torch.nn.functional as F
import artproto as ap

ap.CONFIGS.update({
    'PPN-FA-Correia': dict(lam_fa=1e-3, fa_mode='correia'),
    'Ours+Correia': dict(lam_art=1.0, lam_les=1.0, push='constrained', lam_fa=1e-3, fa_mode='correia'),
})
STATE = {'mode': None, 'mL': None}
_cell_fractions = ap.cell_fractions
_fa_cell = ap.fine_annotation_loss


def cell_fractions(mL, mA, hw):
    STATE['mL'] = mL
    return _cell_fractions(mL, mA, hw)


def fine_annotation_loss(out, y, model, cL, valid):
    mode = STATE['mode']
    if mode is None:
        return _fa_cell(out, y, model, cL, valid)
    outside = (STATE['mL'].float() < 0.5).float()
    sim = out['sim'].float()
    own = (model.proto_class[None, :] == y[:, None]).float()
    if mode == 'correia':
        up = F.adaptive_avg_pool2d(sim, outside.shape[-2:])
        per = (torch.linalg.vector_norm(up * outside, dim=(2, 3)) * own).sum(1)
    else:
        raise ValueError(mode)
    return per[valid].sum() if valid.any() else sim.sum() * 0


ap.cell_fractions = cell_fractions
ap.fine_annotation_loss = fine_annotation_loss


def train_run(cfg, *args, **kw):
    STATE['mode'] = cfg.get('fa_mode')
    return ap.train_run(cfg, *args, **kw)

In [ ]:
%%writefile train6.py
import argparse, os, sys, time, traceback, pandas as pd, torch
import artproto as ap, fa_ext as fx
p = argparse.ArgumentParser()
p.add_argument('--jobs', required=True)
p.add_argument('--deadline', type=float, default=0)
p.add_argument('--out', default='/kaggle/working/runs')
p.add_argument('--smoke', type=int, default=0)
p.add_argument('--bs', type=int, default=0)
p.add_argument('--workers', type=int, default=3)
p.add_argument('--root', default='/kaggle')
a = p.parse_args()
p1, p2 = ap.find_prep_dirs(ap.walk_files(f'{a.root}/input') + ap.walk_files(f'{a.root}/working'))
if not (p1 and p2): sys.exit('ABORT: prepared data not found')
meta = pd.read_csv(f'{p2}/meta.csv')
if not a.smoke:
    assert meta.split.value_counts().to_dict() == {'train': 20552, 'val': 2459, 'test': 2320}, 'split differs from the paper'
prep = dict(img_path=f'{p1}/images.npy', msk_path=f'{p2}/masks.npy', meta=meta, pretrained=not a.smoke)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('data:', p1, '| device:', torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu', flush=True)
claim_dir = f'{a.out}/claims'; os.makedirs(claim_dir, exist_ok=True)
durations = []
for job in a.jobs.split(','):
    name, seed = job.split(':')
    out = f'{a.out}/{name}_s{seed}'
    if os.path.exists(f'{out}/model.pt'):
        continue
    est = 1.15 * max(durations) if durations else 45 * 60
    if a.deadline and time.time() + est > a.deadline:
        print(f'DEADLINE: not starting {job}', flush=True); continue
    try:
        os.close(os.open(f'{claim_dir}/{name}_s{seed}', os.O_CREAT | os.O_EXCL | os.O_WRONLY))
    except FileExistsError:
        continue
    t0 = time.time(); cfg = ap.full_cfg(name, int(seed))
    if a.bs: cfg['bs'] = a.bs
    print(f'=== START {job} | lam_les={cfg["lam_les"]} lam_art={cfg["lam_art"]} push={cfg["push"]} lam_fa={cfg["lam_fa"]} fa_mode={cfg.get("fa_mode")} ===', flush=True)
    try:
        fx.train_run(cfg, prep, out, dev, smoke=bool(a.smoke), workers=a.workers, log=lambda s: print(s, flush=True))
        durations.append(time.time() - t0)
    except Exception:
        traceback.print_exc(); sys.stdout.flush()
print('QUEUE FINISHED for this GPU', flush=True)

## 1. Training of Ours+Correia (three seeds on two GPUs)

In [ ]:
import os, subprocess, time, torch, glob
T_NOTEBOOK = time.time()
jobs = ['Ours+Correia:0', 'Ours+Correia:1', 'Ours+Correia:2']
DEADLINE = T_NOTEBOOK + 6.0 * 3600
ngpu = max(1, torch.cuda.device_count())
os.makedirs('/kaggle/working/runs', exist_ok=True)
FOUND = sorted(glob.glob('/kaggle/input/**/Ours+Correia_s*/model.pt', recursive=True))
print('trained Ours+Correia runs in the inputs:', len(FOUND), *FOUND, sep='\n  ')
procs = []
for g in (range(ngpu) if len(FOUND) < 3 else []):
    log = open(f'/kaggle/working/runs/gpu{g}.log', 'a')
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    procs.append((subprocess.Popen(['python', 'train6.py', '--jobs', ','.join(jobs), '--deadline', str(DEADLINE)],
                                   stdout=log, stderr=subprocess.STDOUT, env=env), g))
print('GPUs:', ngpu, '| jobs:', jobs, flush=True)
t0 = time.time()
while any(pr.poll() is None for pr, _ in procs):
    time.sleep(300)
    print(f'--- {(time.time() - t0) / 60:.0f} min ---', flush=True)
    for _, g in procs:
        lines = open(f'/kaggle/working/runs/gpu{g}.log').read().strip().splitlines()
        print(f'[GPU{g}]', *lines[-2:], sep='\n  ', flush=True)
print('TRAINING DONE in', round((time.time() - t0) / 60, 1), 'min' if procs else 'min (no training: runs attached as input)')
for _, g in procs:
    for line in open(f'/kaggle/working/runs/gpu{g}.log'):
        if any(k in line for k in ('START', 'DONE', 'after push', 'Error', 'error', 'DEADLINE', 'ABORT')): print(f'[GPU{g}]', line.rstrip())

## 2. Evaluation setup

In [ ]:
import os, io, json, time, warnings, urllib.request, numpy as np, pandas as pd, torch, cv2
import torch.nn.functional as F
import importlib, artproto as ap, evalkit as ek; importlib.reload(ap); importlib.reload(ek)
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 500)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
T0 = time.time()
N_IMAGES = 25331
SPLIT_SIZES = {'train': 20552, 'val': 2459, 'test': 2320}
B_BOOT = 2000
N_SEEDS = 3
FILES = ap.walk_files('/kaggle/input') + ap.walk_files('/kaggle/working')
P1, P2 = ap.find_prep_dirs(FILES); PM = P2 or P1
RUNS = ek.find_runs(FILES)
cfg_of = lambda r: r.rsplit('_s', 1)[0]
COUNT = pd.Series([cfg_of(r) for r in RUNS]).value_counts().to_dict()
print('runs per configuration:', COUNT)
for c in ('Ours', 'Ours-stdPush', 'PPN-FA-Correia'):
    assert COUNT.get(c, 0) == N_SEEDS, f'{c}: {COUNT.get(c, 0)} runs'
NEWC = 'Ours+Correia'
HAVE_NEW = COUNT.get(NEWC, 0) > 0
OUT = '/kaggle/working/nb08'; os.makedirs(OUT, exist_ok=True)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
meta = pd.read_csv(f'{PM}/meta.csv')
IM = np.load(f'{P1}/images.npy', mmap_mode='r'); MS = np.load(f'{PM}/masks.npy', mmap_mode='r')
assert len(meta) == IM.shape[0] == MS.shape[0] == N_IMAGES
y = meta.label.values.astype(int)
IDX = {s: np.where(meta.split.values == s)[0] for s in ('train', 'val', 'test')}
assert {k: len(v) for k, v in IDX.items()} == SPLIT_SIZES
TR, VA, TEST = IDX['train'], IDX['val'], IDX['test']
n1 = y[TR].sum(); n0 = len(TR) - n1
CLASS_W = [len(TR) / (2 * n0), len(TR) / (2 * n1)]
GROUPS = meta.group.values[TEST]
VALID = meta.lesion_valid.values.astype(bool)
EXT = {}
for name in ('Derm7pt-derm', 'Derm7pt-clinic', 'PAD-UFES-20'):
    f = f'{PM}/ext_{name}_meta.csv'
    if os.path.exists(f):
        em = pd.read_csv(f)
        EXT[name] = dict(meta=em, keep=em.label.values >= 0, IM=np.load(f'{P1}/ext_{name}_images.npy', mmap_mode='r'),
                         MS=np.load(f'{PM}/ext_{name}_masks.npy', mmap_mode='r'))
pad = EXT['PAD-UFES-20']['meta']
PAD_GROUPS = pad.image.astype(str).str.replace('.png', '', regex=False).str.rsplit('_', n=1).str[0].values
URL = 'https://raw.githubusercontent.com/alceubissoto/debiasing-skin/master/artefacts-annotation/isic_bias.csv'
try:
    ANN = pd.read_csv(io.StringIO(urllib.request.urlopen(URL, timeout=60).read().decode()), sep=';', index_col=0)
except Exception as e:
    loc = [f for f in FILES if f.endswith('isic_bias.csv')]
    ANN = pd.read_csv(loc[0], sep=';', index_col=0) if loc else None
    print('manual labels from', loc[:1], '| download error:', e)
if ANN is not None:
    ANN['base'] = ANN.image.astype(str).str[:12]
    ANN = ANN.drop_duplicates('base').set_index('base')
BASE = meta.image.astype(str).str[:12].values
print('data:', P1, '| device:', dev, '| new runs:', COUNT.get(NEWC, 0), '| manual labels:', None if ANN is None else len(ANN))

In [ ]:
# helpers: latent maps, similarities exactly as ProtoPNet.forward, re-projection from cached latents
@torch.no_grad()
def latents(model, imgs, msks, idx, bs=128):
    Z, CL, CA = [], [], []
    for s in range(0, len(idx), bs):
        ii = idx[s:s + bs]
        x, mL, mA = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(imgs[ii])).to(dev),
                                   torch.from_numpy(np.ascontiguousarray(msks[ii])).to(dev), 'eval')
        with torch.autocast('cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
            z = model.latent(x.contiguous(memory_format=torch.channels_last))
        cL, cA = ap.cell_fractions(mL, mA, z.shape[-2:])
        Z.append(z.float().cpu()); CL.append(cL[:, 0].float().cpu()); CA.append(cA[:, 0].float().cpu())
    return torch.cat(Z), torch.cat(CL), torch.cat(CA)

@torch.no_grad()
def sims_from_z(Z, protos, bs=512, keep_maps=True):
    p = protos.to(dev).float(); p2 = (p ** 2).sum((1, 2, 3)).view(1, -1, 1, 1)
    S, A = [], []
    for s in range(0, len(Z), bs):
        z = Z[s:s + bs].to(dev)
        d = F.relu((z ** 2).sum(1, keepdim=True) + p2 - 2 * F.conv2d(z, p))
        sim = torch.log((d + 1) / (d + 1e-4))
        if keep_maps: S.append(sim.half().cpu())
        A.append(sim.amax((2, 3)).cpu())
    return (torch.cat(S).numpy() if keep_maps else None), torch.cat(A).numpy()

@torch.no_grad()
def repush(Ztr, CLtr, CAtr, ytr, vtr, protos, proto_class, constrained, les_min, art_max, bs=512):
    P, D = protos.shape[:2]
    p = protos.to(dev).float(); p2 = (p ** 2).sum((1, 2, 3)).view(1, -1, 1, 1)
    best_d = torch.full((P,), float('inf'), device=dev); best = [None] * P
    pc = proto_class.to(dev); hw = Ztr.shape[-1] * Ztr.shape[-2]; W_ = Ztr.shape[-1]
    for s in range(0, len(Ztr), bs):
        z = Ztr[s:s + bs].to(dev); B = z.shape[0]
        d = F.relu((z ** 2).sum(1, keepdim=True) + p2 - 2 * F.conv2d(z, p))
        ok = torch.ones(B, 1, *z.shape[-2:], dtype=torch.bool, device=dev)
        if constrained:
            ok = (CLtr[s:s + bs].to(dev)[:, None] >= les_min) & (CAtr[s:s + bs].to(dev)[:, None] <= art_max) \
                 & torch.as_tensor(vtr[s:s + bs], device=dev).view(B, 1, 1, 1)
        own = pc.view(1, P, 1, 1) == torch.as_tensor(ytr[s:s + bs], device=dev).view(B, 1, 1, 1)
        dd = d.masked_fill(~(own & ok), float('inf')).permute(1, 0, 2, 3).reshape(P, -1)
        val, arg = dd.min(1)
        for j in torch.nonzero(val < best_d).flatten().tolist():
            a = int(arg[j]); b, rem = divmod(a, hw); hh, ww = divmod(rem, W_)
            best_d[j] = val[j]; best[j] = (s + b, hh, ww)
    newp = protos.clone().float()
    rows = []
    for j, bj in enumerate(best):
        if bj is None:
            rows.append(dict(proto=j, pos=-1, h=-1, w=-1, les_frac=np.nan, art_frac=np.nan)); continue
        i, h, w = bj; newp[j, :, 0, 0] = Ztr[i, :, h, w]
        rows.append(dict(proto=j, pos=i, h=h, w=w, les_frac=float(CLtr[i, h, w]), art_frac=float(CAtr[i, h, w])))
    return newp, pd.DataFrame(rows)

def auc_fast(yb, p):
    r = rankdata(p); n_1 = yb.sum(); n_0 = len(yb) - n_1
    return (r[yb == 1].sum() - n_1 * (n_1 + 1) / 2) / (n_1 * n_0)
def boot_delta(yb, pa, pb, groups, B=B_BOOT, seed=0):
    rs = np.random.RandomState(seed); u, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(u))]; d = []
    for _ in range(B):
        ii = np.concatenate([members[k] for k in rs.randint(0, len(u), len(u))]); yy = yb[ii]
        if 0 < yy.sum() < len(yy): d.append(auc_fast(yy, pa[ii]) - auc_fast(yy, pb[ii]))
    d = np.array(d)
    if len(d) == 0: return np.nan, np.nan, np.nan
    lo, hi = np.percentile(d, [2.5, 97.5])
    return lo, hi, float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
def gmean(v, groups): return pd.Series(v).groupby(groups).mean()
def group_wilcoxon(va, vb, groups):
    a, b = gmean(va, groups), gmean(vb, groups); ok = np.isfinite(a.values) & np.isfinite(b.values)
    return ek.paired_wilcoxon(a.values[ok], b.values[ok])
def msd(v): v = np.asarray(v, float); return f'{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}'
keep = VALID[TEST]; MS_test = np.array(MS[TEST]); CF_IDX = TEST[keep]
pos_of = {g: k for k, g in enumerate(TEST)}; loc = np.array([pos_of[g] for g in CF_IDX])
CFD = {}
for kind in ('add_dark_corners', 'add_ruler'):
    CF, CH, ok = ek.make_counterfactuals(IM, MS, CF_IDX, kind, seed=0)
    sel = np.where(ok)[0]; CFD[kind] = dict(CF=CF[sel], CH=CH[sel], sel=sel, Ms=np.array(MS[CF_IDX[sel]]))
    del CF, CH
print('counterfactual sets:', {k: len(v['sel']) for k, v in CFD.items()}, '| helpers ready', flush=True)

## 3. Projection thresholds: re-projection of the Ours-stdPush models (no retraining)

In [ ]:
VARIANTS = [('no constraint (= Ours-stdPush)', False, 0.0, 1.0),
            ('lesion >= 0.3, artifact <= 0.05', True, 0.3, 0.05),
            ('lesion >= 0.5, artifact <= 0.05 (paper)', True, 0.5, 0.05),
            ('lesion >= 0.7, artifact <= 0.05', True, 0.7, 0.05),
            ('lesion >= 0.9, artifact <= 0.05', True, 0.9, 0.05),
            ('lesion >= 0.5, artifact = 0', True, 0.5, 0.0),
            ('lesion >= 0.5, artifact <= 0.2', True, 0.5, 0.2),
            ('lesion >= 0.5, no artifact limit', True, 0.5, 1.0)]
ytr, vtr, yte = y[TR], VALID[TR], y[TEST]
def evaluate_protos(model, protos, Ztr, Zva, Zte, Zcf, tag):
    _, Atr = sims_from_z(Ztr, protos, keep_maps=False)
    W, _ = ek.refit_last_layer(model, Atr, ytr, CLASS_W); W = W.cpu().numpy()
    _, Ava = sims_from_z(Zva, protos, keep_maps=False); pva = ek.probs_from_act(Ava, W); thr = ek.youden_threshold(y[VA], pva)
    Ste, Ate = sims_from_z(Zte, protos); pte = ek.probs_from_act(Ate, W)
    at = ek.attribution(Ste[keep], Ate[keep], MS_test[keep], dev)
    r = dict(tag=tag, auc=roc_auc_score(yte, pte), lesion_share=at.hit_L.mean(),
             artifact_share=np.nanmean(np.where(at.area_A.values > 0.005, at.hit_A.values, np.nan)), pointing=at.point_A.mean())
    for kind, cf in Zcf.items():
        Scf, Acf = sims_from_z(cf, protos); p1 = ek.probs_from_act(Acf, W); p0 = pte[loc[CFD[kind]['sel']]]
        r[f'flip {kind[4:]}'] = float(((p0 >= thr) != (p1 >= thr)).mean())
        st = ek.stability(Ste[loc[CFD[kind]['sel']]], Ate[loc[CFD[kind]['sel']]], Scf, Acf, CFD[kind]['CH'], CFD[kind]['Ms'], dev)
        r[f'capture {kind[4:]}'] = float(st.capture.mean()); r[f'rho {kind[4:]}'] = float(st.spearman_top1.mean())
    return r, pte
SWEEP_ROWS, ENS = [], {}
for cfgname in ('Ours-stdPush', 'Ours'):
    for run in sorted(r for r in RUNS if cfg_of(r) == cfgname):
        t = time.time(); model = ek.load_run(RUNS[run], dev)[0]
        Ztr, CLtr, CAtr = latents(model, IM, MS, TR); Zva = latents(model, IM, MS, VA)[0]; Zte = latents(model, IM, MS, TEST)[0]
        Zcf = {k: latents(model, v['CF'], v['Ms'], np.arange(len(v['sel'])))[0] for k, v in CFD.items()}
        _, a_chk = sims_from_z(Zte[:128], model.prototypes.data, keep_maps=False)
        a_ref = ek.forward(model, IM, MS, TEST[:128], dev)['act']          # same batch shape (128) as in latents()
        dif = np.abs(a_chk - a_ref)
        print(f'{run}: cached vs. model activations: max |diff| {dif.max():.2e}, mean {dif.mean():.2e} (activations {a_ref.min():.2f}..{a_ref.max():.2f})', flush=True)
        assert dif.mean() < 5e-2, 'cached similarities differ from the model'   # fp16 noise is far smaller; a logic error gives ~0.1-1
        variants = VARIANTS if cfgname == 'Ours-stdPush' else [('trained with the constraint (Ours)', None, 0.5, 0.05)]
        for name, constrained, lmin, amax in variants:
            if constrained is None:
                protos = model.prototypes.data.clone().float()
                info = pd.read_csv(f'{RUNS[run]}/push_info.csv').rename(columns={'les_frac': 'les_frac', 'art_frac': 'art_frac'})
            else:
                protos, info = repush(Ztr, CLtr, CAtr, ytr, vtr, model.prototypes.data, model.proto_class, constrained, lmin, amax)
            r, pte = evaluate_protos(model, protos, Ztr, Zva, Zte, Zcf, name)
            r.update(run=run, lesion_protos=int((info.les_frac >= 0.5).sum()), mean_cell_lesion=float(info.les_frac.mean()),
                     artifact_protos=int((info.art_frac > 0).sum()), missing=int(info.les_frac.isna().sum()))
            SWEEP_ROWS.append(r); ENS.setdefault(name, []).append(pte)
        del model, Ztr, CLtr, CAtr, Zva, Zte, Zcf; torch.cuda.empty_cache()
        print(f'{run}: {len(variants)} variants | {time.time() - t:.0f}s', flush=True)
SW = pd.DataFrame(SWEEP_ROWS); SW.to_csv(f'{OUT}/threshold_sweep_per_run.csv', index=False)
cols = ['auc', 'lesion_protos', 'mean_cell_lesion', 'artifact_protos', 'lesion_share', 'artifact_share', 'pointing',
        'flip dark_corners', 'flip ruler', 'capture dark_corners', 'capture ruler', 'rho dark_corners']
g = SW.groupby('tag', sort=False)[cols]
SWT = g.mean().round(4)
SWT['auc'] = SW.groupby('tag', sort=False)['auc'].apply(lambda v: msd(v))
SWT['ens_auc'] = [round(roc_auc_score(yte, np.mean(ENS[t], 0)), 4) for t in SWT.index]
ref = np.mean(ENS['lesion >= 0.5, artifact <= 0.05 (paper)'], 0)
SWT['p_vs_paper_thresholds'] = [np.nan if t == 'lesion >= 0.5, artifact <= 0.05 (paper)' else boot_delta(yte, np.mean(ENS[t], 0), ref, GROUPS)[2] for t in SWT.index]
print('\nProjection-threshold sweep (mean over 3 seeds; re-projection of Ours-stdPush unless noted):')
print(SWT.to_string())
print('paper (Tables IV, V, VII): Ours-stdPush AUC .863 +- .005, 7.7 lesion prototypes, artifact share .086, flip DC 4.5%, capture DC 7.5%, rho DC .923;'
      ' Ours AUC .865 +- .006, 20.0, .048, 5.2%, 3.4%, .918')
SWT.to_csv(f'{OUT}/threshold_sweep.csv')

## 4. Ours+Correia: lesion attraction and constrained projection combined with the original mask loss

In [ ]:
CMP_CFG = ['Ours', 'PPN-FA-Correia'] + ([NEWC] if HAVE_NEW else [])
PRED = {}
for run in sorted(r for r in RUNS if cfg_of(r) in CMP_CFG):
    t = time.time(); model, cfg, L = ek.load_run(RUNS[run], dev); R = {}
    for s in ('train', 'val', 'test'):
        o = ek.forward(model, IM, MS, IDX[s], dev, keep_maps=(s == 'test')); R[s] = o
    for s, e in EXT.items():
        R[s] = ek.forward(model, e['IM'], e['MS'], np.arange(len(e['meta'])), dev)
    W, _ = ek.refit_last_layer(model, R['train']['act'], y[TR], CLASS_W); W = W.cpu().numpy(); R.pop('train')
    for s in ['val', 'test'] + list(EXT): R[s]['p'] = ek.probs_from_act(R[s]['act'], W)
    R['thr'] = ek.youden_threshold(y[VA], R['val']['p']); R['W'] = W; R['push'] = pd.read_csv(f'{RUNS[run]}/push_info.csv')
    R['minutes'] = L.get('minutes'); PRED[run] = R
    del model; torch.cuda.empty_cache(); print(f'{run:20s} test AUC {roc_auc_score(y[TEST], R["test"]["p"]):.4f} | {time.time() - t:.0f}s', flush=True)
ylab = lambda s: y[TEST] if s == 'test' else EXT[s]['meta'].label.values
kp = lambda s: np.ones(len(TEST), bool) if s == 'test' else EXT[s]['keep']
ens = lambda c, s: np.mean([PRED[r][s]['p'] for r in PRED if cfg_of(r) == c], 0)
rows = []
for c in CMP_CFG:
    rs_ = [r for r in PRED if cfg_of(r) == c]
    rec = dict(config=c, seeds=len(rs_), protos_on_lesion=np.mean([(PRED[r]['push'].les_frac >= 0.5).sum() for r in rs_]),
               train_minutes=[PRED[r]['minutes'] for r in rs_])
    for s in ['test'] + list(EXT):
        rec[f'AUC {s}'] = msd([roc_auc_score(ylab(s)[kp(s)], PRED[r][s]['p'][kp(s)]) for r in rs_])
    rec['ens AUC test'] = round(roc_auc_score(y[TEST], ens(c, 'test')), 4)
    rows.append(rec)
CLS = pd.DataFrame(rows); print(CLS.to_string(index=False))
rows = []
if HAVE_NEW:
    for c in ('Ours', 'PPN-FA-Correia'):
        for s in ['test'] + list(EXT):
            k = kp(s); yy = ylab(s)[k]; pa, pb = ens(NEWC, s)[k], ens(c, s)[k]
            grp = GROUPS if s == 'test' else (PAD_GROUPS[k] if s == 'PAD-UFES-20' else None)
            lo, hi, pb_ = boot_delta(yy, pa, pb, grp) if grp is not None else (np.nan, np.nan, np.nan)
            rows.append(dict(set=s, versus=c, ens_auc_new=roc_auc_score(yy, pa), ens_auc_versus=roc_auc_score(yy, pb),
                             delta_CI95=f'[{lo:.3f}, {hi:.3f}]', p_group_bootstrap=pb_, p_DeLong=ek.delong(yy, pa, pb)[3]))
    CMP = pd.DataFrame(rows); print('\nOurs+Correia versus Ours and Correia (seed ensembles):'); print(CMP.round(4).to_string(index=False))
# attribution
gk = GROUPS[keep]
PI = {r: ek.attribution(PRED[r]['test']['sim'][keep], PRED[r]['test']['act'][keep], MS_test[keep], dev) for r in PRED}
MET = {'lesion share': lambda df: df.hit_L.values,
       'artifact share': lambda df: np.where(df.area_A.values > 0.005, df.hit_A.values, np.nan),
       'pointing': lambda df: df.point_A.values,
       'DC share': lambda df: np.where(df.area_DC.values > 0.005, df.hit_DC.values, np.nan)}
rows = []; ATT = {}
for m, f in MET.items():
    V = {c: np.nanmean(np.stack([f(PI[r]) for r in PI if cfg_of(r) == c]), 0) for c in CMP_CFG}
    for c in CMP_CFG: ATT[(m, c)] = np.nanmean(V[c])
    if HAVE_NEW:
        for c in ('Ours', 'PPN-FA-Correia'):
            rows.append(dict(metric=m, versus=c, mean_new=np.nanmean(V[NEWC]), mean_versus=np.nanmean(V[c]), p_group=group_wilcoxon(V[NEWC], V[c], gk)))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.3g}'))
# counterfactuals
rows = []; CFM = {}
for kind in ('add_dark_corners', 'add_ruler', 'add_hair', 'add_ink'):
    if kind in CFD:
        CF, CH, sel, Ms = CFD[kind]['CF'], CFD[kind]['CH'], CFD[kind]['sel'], CFD[kind]['Ms']
    else:
        CF, CH, ok = ek.make_counterfactuals(IM, MS, CF_IDX, kind, seed=0); sel = np.where(ok)[0]; CF, CH = CF[sel], CH[sel]; Ms = np.array(MS[CF_IDX[sel]])
    gsel = GROUPS[loc[sel]]; vals = {'flip': {}, 'absdp': {}, 'capture': {}, 'rho': {}}
    for run in PRED:
        R = PRED[run]; model = ek.load_run(RUNS[run], dev)[0]
        o = ek.forward(model, CF, Ms, np.arange(len(sel)), dev, keep_maps=True)
        p1 = ek.probs_from_act(o['act'], R['W']); p0 = R['test']['p'][loc[sel]]
        vals['flip'][run] = ((p0 >= R['thr']) != (p1 >= R['thr'])).astype(float); vals['absdp'][run] = np.abs(p1 - p0)
        if kind in ('add_dark_corners', 'add_ruler'):
            st = ek.stability(R['test']['sim'][loc[sel]], R['test']['act'][loc[sel]], o['sim'], o['act'], CH, Ms, dev)
            vals['capture'][run] = st.capture.values; vals['rho'][run] = st.spearman_top1.values
        del model; torch.cuda.empty_cache()
    for m, D in vals.items():
        if not D: continue
        V = {c: np.mean([D[r] for r in D if cfg_of(r) == c], 0) for c in CMP_CFG}
        for c in CMP_CFG: CFM[(kind, m, c)] = V[c].mean()
        if HAVE_NEW:
            for c in ('Ours', 'PPN-FA-Correia'):
                rows.append(dict(intervention=kind, metric=m, versus=c, mean_new=V[NEWC].mean(), mean_versus=V[c].mean(), p_group=group_wilcoxon(V[NEWC], V[c], gsel)))
    print(f'{kind} done', flush=True)
CFT = pd.DataFrame(rows); print(CFT.to_string(index=False, float_format=lambda v: f'{v:.3g}'))
# real artifacts
tv = meta.iloc[TEST]; ben = y[TEST] == 0; rows = []
seed_dec = lambda c: np.mean([(PRED[r]['test']['p'] >= PRED[r]['thr']).astype(float) for r in PRED if cfg_of(r) == c], 0)
for k in ('DC', 'SP'):
    a = tv[f'art_{k}'].values.astype(bool); sel_b = np.where(ben)[0]; zb = a[sel_b].astype(int)
    for c in CMP_CFG:
        D = seed_dec(c); r = dict(artifact=k, config=c, A=auc_fast(zb, ens(c, 'test')[sel_b]), FPR_with=D[ben & a].mean(), FPR_without=D[ben & ~a].mean())
        if HAVE_NEW and c != NEWC:
            lo, hi, p = boot_delta(zb, ens(NEWC, 'test')[sel_b], ens(c, 'test')[sel_b], GROUPS[sel_b]); r.update(delta_new_minus_c=f'[{lo:.3f}, {hi:.3f}]', p_group=p)
        rows.append(r)
REAL = pd.DataFrame(rows); print(REAL.round(4).to_string(index=False))
SUMR = []
for c in CMP_CFG:
    SUMR.append({'config': c, 'AUC': CLS[CLS.config == c]['AUC test'].iloc[0], 'Derm7pt-derm': CLS[CLS.config == c]['AUC Derm7pt-derm'].iloc[0],
                 'Derm7pt-clin': CLS[CLS.config == c]['AUC Derm7pt-clinic'].iloc[0], 'PAD': CLS[CLS.config == c]['AUC PAD-UFES-20'].iloc[0],
                 'protos on lesion': CLS[CLS.config == c].protos_on_lesion.iloc[0], 'artifact share': round(ATT[('artifact share', c)], 4),
                 'lesion share': round(ATT[('lesion share', c)], 4), 'flip DC %': round(100 * CFM[('add_dark_corners', 'flip', c)], 2),
                 'flip ruler %': round(100 * CFM[('add_ruler', 'flip', c)], 2), 'flip hair %': round(100 * CFM[('add_hair', 'flip', c)], 2),
                 'flip ink %': round(100 * CFM[('add_ink', 'flip', c)], 2), 'rho DC': round(CFM[('add_dark_corners', 'rho', c)], 4),
                 'capture DC %': round(100 * CFM[('add_dark_corners', 'capture', c)], 2),
                 'A real DC': round(REAL[(REAL.artifact == 'DC') & (REAL.config == c)].A.iloc[0], 4)})
SUMT = pd.DataFrame(SUMR); print('\nSummary (Ours+Correia vs Ours vs Correia):'); print(SUMT.T.to_string())

## 5. Prototype cells: undetected artifacts, border cells, robustness to lesion-mask errors

In [ ]:
# cell fractions of all training images (model independent)
CLtr_all, CAtr_all = [], []
for s in range(0, len(TR), 256):
    ii = TR[s:s + 256]
    _, mL, mA = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(IM[ii])).to(dev), torch.from_numpy(np.ascontiguousarray(MS[ii])).to(dev), 'eval')
    cL, cA = ap.cell_fractions(mL, mA, (7, 7)); CLtr_all.append(cL[:, 0].cpu().numpy()); CAtr_all.append(cA[:, 0].cpu().numpy())
CLtr_all = np.concatenate(CLtr_all); CAtr_all = np.concatenate(CAtr_all)
admissible = (CLtr_all >= 0.5) & (CAtr_all <= 0.05) & VALID[TR][:, None, None]
border = np.zeros((7, 7), bool); border[[0, -1], :] = True; border[:, [0, -1]] = True
TYPES = {'DC': ['dark_corner'], 'TS': ['hair', 'ruler'], 'SP': ['gel_bubble', 'gel_border'], 'INK': ['ink', 'patches']}
if ANN is not None:
    inb = np.array([b in ANN.index for b in BASE[TR]])
    print(f'training images with manual labels: {inb.sum()} of {len(TR)}; admissible cells in them: {admissible[inb].sum()} '
          f'(of {admissible.sum()} admissible cells overall)')
    rows = []
    for k, cols in TYPES.items():
        man = np.zeros(len(TR), bool); man[inb] = ANN.loc[BASE[TR][inb], cols].max(1).values > 0
        det = meta[f'art_{k}'].values[TR].astype(bool)
        und = inb & man & ~det
        cells_und = admissible[und].sum(); cells_und_border = (admissible[und] & border[None]).sum()
        rows.append(dict(type=k, manual_label=' or '.join(cols), images_manual=int((inb & man).sum()), images_undetected=int(und.sum()),
                         admissible_cells_in_labelled_images=int(admissible[inb].sum()),
                         share_in_undetected_images=cells_und / max(admissible[inb].sum(), 1),
                         share_on_border_of_undetected_images=cells_und_border / max(admissible[inb].sum(), 1)))
    UND = pd.DataFrame(rows); print('\nAdmissible cells (lesion >= 0.5, detected artifact <= 0.05) in images with a manual but undetected artifact:')
    print(UND.round(4).to_string(index=False))
# prototype cells of every prototype configuration
pos_tr = {g: k for k, g in enumerate(TR)}
rows = []
for run, d in RUNS.items():
    f = f'{d}/push_info.csv'
    if not os.path.exists(f): continue
    pi = pd.read_csv(f); pi['tpos'] = pi.img.map(pos_tr)
    r = dict(config=cfg_of(run), run=run, off_lesion=int((pi.les_frac < 0.5).sum()), border_cells=int(((pi.les_frac >= 0.5) & (pi.les_frac < 0.95)).sum()),
             interior_cells=int((pi.les_frac >= 0.95).sum()), mean_cell_lesion=float(pi.les_frac.mean()))
    if ANN is not None:
        inl = pi.tpos.map(lambda t: BASE[TR][int(t)] in ANN.index if t == t else False).astype(bool)
        r['protos_in_labelled_images'] = int(inl.sum())
        for k, cols in TYPES.items():
            und = []
            for _, q in pi[inl].iterrows():
                t = int(q.tpos); manual = ANN.loc[BASE[TR][t], cols].max() > 0; detected = bool(meta[f'art_{k}'].values[TR][t])
                und.append(manual and not detected and (k != 'DC' or border[int(q.h), int(q.w)]))
            r[f'undetected_{k}'] = int(np.sum(und))
    rows.append(r)
PC = pd.DataFrame(rows); PC.to_csv(f'{OUT}/prototype_cells_per_run.csv', index=False)
agg_cols = [c for c in PC.columns if c not in ('config', 'run')]
print('\nPrototype cells per run (mean over seeds; undetected_DC counts border cells only):')
print(PC.groupby('config')[agg_cols].mean().round(2).to_string())
# robustness of the provenance guarantee to lesion-mask errors (+-5 px at 256 px, as for Table X)
def masks_mod(msk, delta):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * abs(delta) + 1, 2 * abs(delta) + 1)); out = np.array(msk).copy()
    for i in range(len(out)):
        L = (out[i] & ap.BIT_L).astype(np.uint8)
        L2 = cv2.dilate(L, k) if delta > 0 else cv2.erode(L, k)
        art = out[i] & ap.BIT_ART
        if delta > 0: art = np.where(L2 > 0, 0, art).astype(np.uint8)
        out[i] = art | (L2 * ap.BIT_L)
    return out
rows = []
for c in ('Ours', 'Ours-noArt', 'Ours-noLes') + ((NEWC,) if HAVE_NEW else ()):
    for run in sorted(r for r in RUNS if cfg_of(r) == c):
        pi = pd.read_csv(f'{RUNS[run]}/push_info.csv')
        for delta in (-5, 0, 5):
            ms = masks_mod(np.array(MS[pi.img.values]), delta) if delta else np.array(MS[pi.img.values])
            _, mL, mA = ap.gpu_augment(torch.from_numpy(np.ascontiguousarray(IM[pi.img.values])).to(dev), torch.from_numpy(ms).to(dev), 'eval')
            cL, cA = ap.cell_fractions(mL, mA, (7, 7)); cl = cL[:, 0].cpu().numpy()
            fr = np.array([cl[k, int(h), int(w)] for k, (h, w) in enumerate(zip(pi.h, pi.w))])
            rows.append(dict(config=c, run=run, delta_px=delta, protos_lesion_ge_05=int((fr >= 0.5).sum()), mean_cell_lesion=float(fr.mean()), min_cell_lesion=float(fr.min())))
ROB = pd.DataFrame(rows); print('\nProvenance guarantee under lesion-mask errors (mean over seeds, of 20 prototypes):')
print(ROB.groupby(['config', 'delta_px'])[['protos_lesion_ge_05', 'mean_cell_lesion', 'min_cell_lesion']].mean().round(3).to_string())

## 6. Summary

In [ ]:
print(f'NB08 evaluation finished in {(time.time() - T0) / 60:.1f} min')
print('\n=== threshold sweep ===\n' + SWT.to_string())
if HAVE_NEW:
    print('\n=== Ours+Correia ===\n' + SUMT.T.to_string())
import subprocess
print(subprocess.run('cd /kaggle/working && zip -qr nb08.zip nb08 2>/dev/null; ls -la nb08', shell=True, capture_output=True, text=True).stdout)
print('NB08 FINISHED')